# Planning for electric vehicle charging sites
# Maximum (Weighted) Independent Set with QoolQit and the MIS solver

### Pasqal Workshop
#### Trainers: Vincent Martin, Krisztian Benyo

In this tutorial you will:

1. Learn what **Maximum Independent Set (MIS)** and its weighted variant **MWIS** are, and why they fit **neutral-atom
   (Rydberg) quantum hardware** so naturally.
2. Meet **[QoolQit](https://github.com/pasqal-io/qoolqit)**, a toolbox for the Rydberg analog model, and the
   **[MIS solver](https://github.com/pasqal-io/maximum-independent-set)** (package `maximum-independent-set`, imported as `mis`),
   the high-level library that can help solve MIS/MWIS for you.
3. Solve a small MWIS instance **by hand with QoolQit** (embedding, detuning-map drive, emulation).
4. Build a **minimum working example of an industrial pipeline**: selecting EV charging sites that do not cannibalise
   each other, maximising expected revenue (a weighted MIS on a geometric conflict graph).

> **Sources:** The presented content follows the QoolQit tutorial
> [*Solving a MWIS problem*](https://docs.pasqal.com/qoolqit/tutorials/Solving_a_MWIS/).
>
> The optional MIS solver elements follow the
> [MIS solver documentation](https://docs.pasqal.com/applicationsolvingtools/mis/).

## Setup

Everything runs **locally on an emulator**; no quantum hardware or cloud account is needed.
Use a fresh virtual environment (Python 3.10+; it is recommended to have a recent 3.11 or 3.12 Python version).

In [ ]:
import itertools
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

Please do keep in mind that both QoolQit and Maximum-Independent-Set libraries are young and evolving.

---
# 1. Introduction

## 1.1 Graphs, MIS and MWIS

A few applications of graphs:

- represent molecules;
- represent roads;
- represent dependencies;
- represent _conflicts_.

In [ ]:
# Create an undirected graph of your choice
example_graph_1 = nx.Graph()

# Add nodes
for i in range(5):
    example_graph_1.add_node(i)

# Add edges
example_graph_1.add_edge(0, 1)
example_graph_1.add_edge(0, 2)
example_graph_1.add_edge(0, 3)
example_graph_1.add_edge(0, 4)
example_graph_1.add_edge(1, 2)

In [ ]:
nx.draw(example_graph_1)

In [ ]:
example_graph_2 = nx.Graph()
CLIQUE_NODES = [f"clique_{i}" for i in range(10)]
INDIVIDUAL_NODES = [f"individual_{i}" for i in range(10)]

example_graph_2.add_nodes_from(CLIQUE_NODES)
example_graph_2.add_nodes_from(INDIVIDUAL_NODES)

for i, node in enumerate(CLIQUE_NODES):
    for node_2 in CLIQUE_NODES[i+1:]:
        example_graph_2.add_edge(node, node_2)

for node in INDIVIDUAL_NODES:
    example_graph_2.add_edge(CLIQUE_NODES[0], node)
    example_graph_2.add_edge(CLIQUE_NODES[1], node)

nx.draw_kamada_kawai(example_graph_2)

Consider an undirected graph $G(V, E)$ composed of a set of vertices $V$ connected by unweighted edges $E$. An **independent set** of this graph is a subset of vertices where any two elements of this subset are not connected by an edge. Given an undirected graph $G=(V, E)$, a set of nodes $V' \subseteq V$ is _independent_ in $G$ if $V'^2 \cap E = \{\}$.

In [ ]:
nx.draw_kamada_kawai(example_graph_2, nodelist=[INDIVIDUAL_NODES[0], INDIVIDUAL_NODES[1]])

**MaximAL Independent Set** Given a graph $G=(V, E)$, a set of nodes $V'$ is a MaximAL Independent Set if there is no $V''$ such that satisfies all of:
- $V' \subsetneq V''$
- $V'' \subseteq V$
- $V''$ is independent in $G$.

Equivalently a **MaximAL Independent Set** is an independent subset that cannot be expanded such that it still stays independent.

In [ ]:
nx.draw_kamada_kawai(example_graph_2, nodelist=[CLIQUE_NODES[0]])

**MaximUM Independent Set** (MIS) Given a graph $G=(V, E)$, a set of nodes $V'$ is a MaximUM Independent Set if there is no $V''$ such that satisfies all of:
- $|V''| > |V'|$
- $V''$ is independent in $G$.

Equivalently, the Maximum Independent Set (MIS) corresponds to the largest independent subsets, and it is in general an NP-complete problem to determine the MIS of a graph. Solving MIS is $O(3^{\frac{n}{3}})$, i.e. _it quickly grows untractable_. Quantum can run many instances (not all) in _polynomial time_, i.e. exponentially faster. The MIS problem has several interesting applications, such as financial portfolio optimisation, optimization of telecommunications systems, grid optimisation for electric vehicle charging, etc.

In [ ]:
import itertools
nx.draw_kamada_kawai(example_graph_2, nodelist=list(itertools.chain(INDIVIDUAL_NODES, [CLIQUE_NODES[2]])))

**MWIS** or Maximum Weighted Independent Set arises when the graph $G$ has for every vertex $i$ a weight $w_i \ge 0$ and the task is to find an independent set of maximum *total weight* $\sum_{i\in S} w_i$.

MWIS is in general NP-hard, and appears whenever you must choose as many (or as valuable) items as possible while some pairs are
mutually exclusive: placing antennas or charging stations so that they do not interfere, scheduling satellite observations,
selecting non-overlapping jobs, and so on.

**As a QUBO problem:** When looking for the MIS of a graph, we separate the vertices into two distinct classes: an independent set and the rest. We can attribute a status $x$ to each vertex, where $x_i=1$ if vertex $i$ is attributed to the independent set, and $x_i=0$ otherwise. The Maximum Independent Set corresponds to the minima of the following cost function:
$$
C(x_1,\ldots,x_N)=-\sum\limits_{i=1}^N x_i + \lambda \sum\limits_{(i,j)\in E} x_ix_j.
$$
In this cost function, we want to promote a maximal number of vertices to the independent set, so to the $1$ indicator, but the fact that $\lambda$ is a large constant that strongly penalizes two adjacent vertices for being selected at the same time. The minimum of the cost function $C$ therefore corresponds to the Maximum Independent Set.

Similarly, MWIS is the minimisation of
$$
C(x) = -\sum_{i=1}^N w_i\, x_i \;+\; \lambda \sum_{(i,j)\in E} x_i x_j ,
$$
where the penalty $\lambda$ must be large enough (at least about $2\max_i w_i$) that violating an edge never pays off.

## 1.2 Why Rydberg atoms?

The MIS problem on unit disks graphs can be tackled by using an ensemble of interacting cold neutral atoms as a quantum resource, where each atom represents a vertex of the graph under study. As with any quantum system, the dynamics of the atoms are governed by the Schrödinger equation involving a Hamiltonian depending on the atomic positions, the electronic energy levels and their interactions.

Interestingly, the physical interactions encoded in this Ising type Hamiltonian constrain the dynamics to only explore independent sets of the graph under study, then leading to an efficient search in the set of possible solutions. More precisely, the levels of two Rydberg atoms strongly interact if the distance between the atoms is smaller than the Rydberg blockade radius, resulting in the impossibility for the two atoms to be both in the same state at the same time. This naturally corresponds to the independent set constraint in the graph defined by the atoms, with edges linking atoms that sit at a distance closer than the aforementioned blockade radius.

In a neutral-atom quantum processor each qubit is an atom in an optical tweezer. An atom excited to a high energy **Rydberg state**
shifts the energy of its neighbours by an interaction $J_{ij}$ that decays like $1/r_{ij}^6$. Two atoms closer
than the **blockade radius** cannot both be excited, which is exactly the independent-set constraint on the
**unit-disk graph** whose edges connect atoms that are closer than the blockade radius.

$$
H = \sum_i \big(\tfrac{\Omega}{2}\,\sigma^x_i - \delta\, n_i\big) + \sum_{i<j} J_{ij}\, n_i n_j .
$$

* A **global detuning** $\delta$ rewards excitation of every atom: this encodes the *cardinality* (plain MIS).
* For **weights** you need a *different reward per atom*. Pasqal hardware does this with a **Detuning Map Modulator (DMM)**:
  a second detuning pattern that is applied with a different strength on each atom. Using normalised weights
  $\tilde w_i = w_i/\max w$, atoms with a large weight receive a smaller negative shift, so heavier vertices are
  excited (selected) more readily.
* An **adiabatic schedule** slowly raises $\Omega$, sweeps $\delta$ from negative to positive, and lowers $\Omega$ again;
  the system ends in (or near) the ground state of aforementioned Hamiltonian $H$, whose bitstring is the optimal independent set.

Graphs that are (close to) unit-disk graphs are therefore **native** to the hardware; general graphs require an extra consideration on embedding.

## 1.3 QoolQit: the neutral-atom toolbox

[QoolQit](https://github.com/pasqal-io/qoolqit) is a Python library for algorithm development in the **Rydberg analog model**,
leveraging dimensionless units. In this notebook we will use the following elements:

| Building block | Purpose and commands |
|---|---|
| `qoolqit.graphs.DataGraph` | a graph with data attached (node weights, coordinates) |
| `qoolqit.embedding.InteractionEmbedder` | turns an interaction matrix into a graph with node coordinates |
| `Register` | the atom positions (`Register.from_graph`) |
| `Drive`, waveforms (`InterpolatedWaveform`, `ConstantWaveform`), `DetuningMapModulator` | amplitude, global detuning and the weight-dependent detuning map |
| `QuantumProgram` | register and drive combined; `compile_to(device)` checks and rescales it |
| `MockDevice`, `AnalogDeviceWithDMM` | an idealised device and a realistic device with a DMM |
| `qoolqit.execution.LocalEmulator` | runs the program and returns bitstring counts |

## 1.4 The MIS solver: the high-level library

The [MIS solver](https://github.com/pasqal-io/maximum-independent-set) lets you solve Maximum Independent Set problems as a **black box** with different classical and quantum backends. The key idea: give a graph, obtain large independent sets.

| Building Block | Purpose and commands |
|---|---|
| `MISInstance(graph)` | wraps a `networkx.Graph`; node attribute `"weight"` holds weights; `to_qubo()`, `draw()` |
| `SolverConfig(...)` | Key parameters of the command are `backend`, `weighting` (`Weighting.UNWEIGHTED` / `WEIGHTED`), `method` (`MethodType.EAGER` / `GREEDY`), `runs` (shots), `embedder`, `pulse_shaper`, pre/post-processing |
| `BackendConfig(...)` | where to run: local emulators (e.g. `BackendType.QUTIP`, `EMU_MPS`, `EMU_SV`), remote emulators, or a QPU |
| `MISSolver(instance, config).solve()` | runs the pipeline and returns a list of `MISSolution` |
| `MISSolution` | Key attributes of the object are `.nodes`, `.node_indices`, `.size`, `.frequency`, `.draw()` |

Behind `solve()` sits a pipeline: **preprocessing** (kernelization shrinks the graph) -> **embedding** onto atoms -> **pulse shaping**
-> **execution** on the backend -> **postprocessing** (sort and improve the samples). With `backend=None` a classical heuristic
is used instead, which can be useful for establishing a baseline.

| | QoolQit | MIS solver |
|---|---|---|
| Level | low-level Rydberg toolbox | high-level MIS/MWIS pipeline |
| Input | register + drive (+ DMM) | a `networkx` graph |
| You choose | positions, waveforms, device, shots | (optionally) method, weighting, embedder, backend |
| Output | bitstring counts | ranked independent sets with frequencies |

### Helpers

Two small helpers are used throughout the Notebook. 

The brute-force solver is fully provided; checking correctness of a result against it is how we trust the quantum output on small instances.

### Independence and weight

In [ ]:
def is_independent(G, nodes):
    """True if no two nodes in `nodes` are adjacent in G."""
    ok = all(not G.has_edge(u, v) for u, v in itertools.combinations(nodes, 2))
    return bool(ok)


def set_weight(G, nodes):
    """Total weight of a node set (a node without a 'weight' attribute counts as 1)."""
    return float(sum(G.nodes[n].get("weight", 1.0) for n in nodes))

#### A quick exercise 

Test the independence test function on the following simple line graph:

In [ ]:
P4 = nx.path_graph(4)                    # 0 - 1 - 2 - 3
nx.draw(P4)

In [ ]:
is_independent(P4, [0, 1, 3]) # For Example

In [ ]:
# You can try different combinations

### Brute-force MWIS

In [ ]:
def brute_force_mwis(G):
    """Exact MWIS by enumeration of all 2^n subsets (only for small graphs!). Returns (nodes, weight)."""
    nodes = list(G.nodes)
    feasible = []
    for r in range(len(nodes) + 1):
        for subset in itertools.combinations(nodes, r):
            if is_independent(G, subset):
                feasible.append(subset)
    best = max(feasible, key=lambda s: set_weight(G, s))
    return sorted(best), set_weight(G, best)

#### Exercise

Test the brute force algorithm for the following simple graph:

In [ ]:
H = nx.Graph([(0, 1), (1, 2), (2, 3)])
nx.set_node_attributes(H, {0: 1.0, 1: 5.0, 2: 1.0, 3: 4.0}, "weight")

In [ ]:
# You can write your code here

---
# 2. Example - Solving a MWIS with QoolQit

We follow the official QoolQit tutorial. The graph has 4 nodes and edges $0-1$, $0-2$, $0-3$, $2-3$. Nodes $1$ and $2$ have weight 2,
nodes 0 and 3 have weight 0. The optimum is $\{1, 2\}$, i.e. bitstring `0110`, with weight 4.

## 2.1 Graph, classical solution, embedding

In [ ]:
from qoolqit.graphs import DataGraph

graph = DataGraph([(0, 1), (0, 2), (0, 3), (2, 3)])
graph.node_weights = {0: 0.0, 1: 2.0, 2: 2.0, 3: 0.0}

# Classical reference with our brute force (networkx copy with a "weight" attribute).
G_ref = nx.Graph(list(graph.edges))
nx.set_node_attributes(G_ref, {0: 0.0, 1: 2.0, 2: 2.0, 3: 0.0}, "weight")
print("Classical MWIS:", brute_force_mwis(G_ref))

In [ ]:
weights = graph.node_weights
node_color = ["tab:green" if weights[n] > 0 else "tab:blue" for n in graph.nodes]
node_size = [600 + 600 * weights[n] for n in graph.nodes]

graph.draw(node_color=node_color, node_size=node_size, font_weight="bold")

**Embedding.** The `InteractionEmbedder` takes a matrix and returns a graph with node coordinates such that pairwise
interaction strengths reproduce its off-diagonal entries. Following the tutorial, we build a matrix with the node weights on the
diagonal and a $1$ for every edge.

In [ ]:
from qoolqit import Register
from qoolqit.embedding import InteractionEmbedder

node_weights = np.array([graph.node_weights[n] for n in graph.nodes])

Q = np.diag(node_weights)
for i, j in graph.edges:
    Q[i, j] = 1.0
    Q[j, i] = 1.0

embedder = InteractionEmbedder()
embedded_graph = embedder.embed(Q)          # DataGraph with optimised coordinates
embedded_graph.draw(node_color=node_color, node_size=node_size, font_weight="bold")

register = Register.from_graph(embedded_graph)

In [ ]:
print(Q)

## 2.2 The drive: global schedule + detuning map

1. **Energy scales.** The strongest interaction is set by the closest pair of atoms, $J_{\max}=1/d_{\min}^6$. We take
   $\Omega = 10\,J_{\max}$ (safely above the interaction scale), $\delta_0=-\Omega$ and $\delta_f=+\Omega$.
2. **Global waveforms.** Amplitude `0 -> Omega -> 0`; detuning `delta_0 -> 0 -> delta_f`.
3. **DMM for the weights.** The map contains a *normalised* value per atom; the DMM waveform is the constant `-delta_f`.
   With normalised $\tilde w_i = w_i/\max_j w_j$ the DMM weight is $1-\tilde w_i$: **heavy nodes receive almost no extra
   (negative) detuning**, so they are excited more easily than light nodes.

In [ ]:
def min_distance(register, embedded_graph):
    """Smallest pairwise atom distance (falls back to the embedded coordinates)."""
    try:
        return float(register.min_distance())
    except Exception:
        pts = np.array(list(embedded_graph.coords.values()), dtype=float)
        d = np.linalg.norm(pts[:, None, :] - pts[None, :, :], axis=-1)
        return float(d[np.triu_indices(len(pts), k=1)].min())


max_interaction = 1.0 / min_distance(register, embedded_graph) ** 6
Omega = 10.0 * max_interaction     # drive amplitude, safely above the interaction scale
delta_0 = -Omega                   # negative: trivial |gggg> ground state at t = 0
delta_f = -delta_0                 # positive: excitations rewarded at t = T
T = 200.0                          # long schedule: adiabatic on an ideal device
print(f"max interaction = {max_interaction:.3g},  Omega = {Omega:.3g},  T = {T}")

In [ ]:
dmm_weights = 1.0 - node_weights / node_weights.max()

# Map each weight to its atom (the register's qubit ids follow Q's row order).
det_map = {qubit: float(dmm_weights[i]) for i, qubit in enumerate(register.qubits.keys())}
print("DMM weights (epsilon_i):", det_map)

We can now assemble the drive and the program, compile it to an **ideal device** (`MockDevice`) and emulate it.

In [ ]:
from qoolqit import ConstantWaveform, Drive, InterpolatedWaveform
from qoolqit.drive import DetuningMapModulator

dmm = DetuningMapModulator(ConstantWaveform(T, -delta_f), det_map)

drive = Drive(
    amplitude=InterpolatedWaveform(T, [0.0, Omega, 0.0]),
    detuning=InterpolatedWaveform(T, [delta_0, 0.0, delta_f]),
    dmm=dmm,
)

In [ ]:
from qoolqit import MockDevice, QuantumProgram

program = QuantumProgram(register, drive)
program.compile_to(device=MockDevice())
program.draw()

In [ ]:
from qoolqit.execution import LocalEmulator

emulator = LocalEmulator()
job = emulator.run(program)
results = job.results()

counts = results.final_bitstrings
print("Most frequent bitstring:", max(counts, key=counts.get))

### Read off the answer

`counts` maps bitstrings to the number of shots. Here, we extract the **most frequent bitstring** and its probability.

In [ ]:
counts = Counter(counts)
n_shots = sum(counts.values())

best_bitstring = counts.most_common(1)[0][0]

print(f"Most frequent bitstring: {best_bitstring}  ({counts[best_bitstring] / n_shots:.1%} of shots)")
print("Expected optimum 0110 ->", "match" if best_bitstring == "0110" else "no match (try a longer T)")

## 2.3 A realistic device

Real devices limit the pulse duration, so the sweep must be much shorter and less adiabatic. With `AnalogDeviceWithDMM`
(which supports a DMM) the optimum should still be the most frequent outcome, but with a
**lower probability**.

In [ ]:
from qoolqit import AnalogDeviceWithDMM

def build_program(T):
    """Register + drive (global schedule and DMM) for a total duration T."""
    det_map = {qubit: float(dmm_weights[i]) for i, qubit in enumerate(register.qubits.keys())}
    dmm = DetuningMapModulator(ConstantWaveform(T, -delta_f), det_map)
    drive = Drive(
        amplitude=InterpolatedWaveform(T, [0.0, Omega, 0.0]),
        detuning=InterpolatedWaveform(T, [delta_0, 0.0, delta_f]),
        dmm=dmm,
    )
    return QuantumProgram(register, drive)
    
T_analog = 7.5
program_analog = build_program(T_analog)
program_analog.compile_to(device=AnalogDeviceWithDMM())

res_analog = LocalEmulator().run(program_analog).results()
counts_analog = Counter(res_analog.final_bitstrings if hasattr(res_analog, "final_bitstrings")
                        else res_analog[0].final_bitstrings)
n = sum(counts_analog.values())
print("Top outcomes:", [(b, round(c / n, 3)) for b, c in counts_analog.most_common(4)])

---
# 3. Industrial mini-pipeline: EV charging site selection

## 3.1 The business problem

A network operator has **10 candidate sites** for fast-charging hubs. Each site has an **expected annual revenue** (in k€).
Two sites **closer than 1.2 km** would cannibalise each other's customers, so at most one of any such pair should be built.

**Goal:** select the set of sites with maximum total expected revenue with no two selected sites closer than 1.2 km. 

This is exactly **MWIS on the associated conflict graph** (vertices = sites, weights = revenue, edges = sites that are too close).

Because conflicts come from distances, the conflict graph is a **unit-disk graph**: precisely the structure that neutral-atom
registers realise natively. (This is the same pattern as in published MIS case studies on EV charging network planning
and on 5G antenna placement.)

**Pipeline:** `data -> conflict graph -> MWIS instance -> exact reference + classical baseline -> MIS solver (quantum emulation) -> validate/repair -> report`

In [ ]:
sites = pd.DataFrame(
    {
        "site": ["Station", "Mall", "Harbor", "Airport", "Hospital",
                 "University", "Old Town", "Retail Park", "Highway", "Suburb"],
        "x_km": [0.0, 0.8, 1.6, 0.4, 1.3, 2.3, 0.0, 1.0, 2.0, 2.9],
        "y_km": [0.0, 0.3, 0.0, 1.1, 1.2, 0.9, 2.1, 2.2, 2.0, 1.9],
        "revenue_k": [5, 7, 5, 4, 8, 6, 8, 8, 10, 6],
    }
)
RADIUS_KM = 1.2
print(sites.to_string())

### Exercise 1 - the conflict graph

Two sites conflict when their Euclidean distance is **below** `RADIUS_KM`. Complete the condition inside `build_conflict_graph`.

In [ ]:
def build_conflict_graph(sites, radius):
    """Weighted conflict graph: nodes = sites (weight = revenue), edge if two sites are closer than `radius`."""
    G = nx.Graph()
    for i, row in sites.iterrows():
        G.add_node(int(i), weight=float(row.revenue_k), pos=(row.x_km, row.y_km))
    for i, j in itertools.combinations(range(len(sites)), 2):
        dist = float(np.hypot(sites.x_km[i] - sites.x_km[j], sites.y_km[i] - sites.y_km[j]))
        if ...:  # <-- YOUR CODE HERE: sites i and j are too close
            G.add_edge(i, j)
    return G


G = build_conflict_graph(sites, RADIUS_KM)
assert G.number_of_nodes() == 10
assert G.has_edge(0, 1) and G.has_edge(0, 3)          # 0.85 km and 1.17 km apart
assert not G.has_edge(0, 2)                           # 1.6 km apart
print("Exercise 4.1 passed:", G.number_of_edges(), "conflicts ->", sorted(G.edges))

In [ ]:
pos = nx.get_node_attributes(G, "pos")
weights = nx.get_node_attributes(G, "weight")

fig, ax = plt.subplots(figsize=(6.5, 5))
nx.draw_networkx(G, pos, ax=ax, node_color="lightsteelblue", node_size=900,
                 labels={n: f"{n}\n{int(weights[n])}k" for n in G}, font_size=8)
ax.set_title("Conflict graph (label: site id, revenue); edges = closer than 1.2 km")
ax.set_aspect("equal")
plt.show()

## 3.2 Reference solutions

We first need (i) the **exact optimum** (here by brute force over $2^{10}=1024$ subsets) and (ii) a
**simple classical baseline**: a greedy heuristic that repeatedly takes the most valuable remaining site and discards its conflicting
neighbours.

### Exercise 2 - greedy baseline

Complete the line that orders the candidate sites by **decreasing revenue**.

In [ ]:
def greedy_mwis(G):
    """Greedy MWIS: repeatedly pick the heaviest remaining node and delete its neighbours."""
    remaining = ...  # <-- YOUR CODE HERE: list of nodes sorted by decreasing weight
    chosen = []
    while remaining:
        n = remaining.pop(0)
        chosen.append(n)
        remaining = [m for m in remaining if not G.has_edge(n, m)]
    return sorted(chosen)


greedy_nodes = greedy_mwis(G)
assert is_independent(G, greedy_nodes)
print("Greedy sites:", greedy_nodes, "-> revenue", set_weight(G, greedy_nodes))

In [ ]:
exact_nodes, exact_weight = brute_force_mwis(G)
greedy_weight = set_weight(G, greedy_nodes)
print(f"Exact optimum : {exact_nodes}  revenue = {exact_weight:.0f}k")
print(f"Greedy        : {greedy_nodes}  revenue = {greedy_weight:.0f}k  "
      f"({greedy_weight / exact_weight:.1%} of optimum)")

## 3.3 Solve the problem

We now solve the charging-site MWIS **with QoolQit**, repeating the recipe of Section 2 on the 10-site conflict graph `G`.

In [ ]:
from qoolqit import DataGraph, Register
from qoolqit.embedding import SpringLayoutEmbedder

n_sites = G.number_of_nodes()
site_weights = np.array([G.nodes[n]["weight"] for n in range(n_sites)])

G_site = DataGraph.from_nx(G)
site_graph = SpringLayoutEmbedder().embed(G_site)     # DataGraph with optimised coordinates
site_graph.draw()
site_register = Register.from_graph(site_graph)

In [ ]:
from qoolqit import ConstantWaveform, Drive, InterpolatedWaveform
from qoolqit.drive import DetuningMapModulator
from qoolqit import MockDevice, AnalogDeviceWithDMM, QuantumProgram

J_max = 1.0 / min_distance(site_register, site_graph) ** 6
Omega_s = 0.5 * J_max            # amplitude tailored to the interaction scale
delta_0_s = -Omega_s              # negative: all atoms in the ground state at t = 0
delta_f_s = -delta_0_s            # positive: excitations rewarded at t = T
T_s = 200.0                       # long schedule (ideal device)

# Detuning-map weights: same normalisation as in Section 2
site_dmm_weights = 1.0 - site_weights / site_weights.max()
det_map = {q: float(site_dmm_weights[i]) for i, q in enumerate(site_register.qubits.keys())}
dmm = DetuningMapModulator(ConstantWaveform(T_s, -delta_f_s), det_map)

site_drive = Drive(
    amplitude=InterpolatedWaveform(T_s, [0.0, Omega_s, 0.0]),
    detuning=InterpolatedWaveform(T_s, [delta_0_s, 0.0, delta_f_s]),
    dmm=dmm,
)

site_program = QuantumProgram(site_register, site_drive)
site_program.draw()
site_program.compile_to(device=MockDevice())

In [ ]:
from qoolqit.execution import LocalEmulator

try:
    site_emulator = LocalEmulator(num_shots=1000)
except TypeError:
    site_emulator = LocalEmulator()
site_res = site_emulator.run(site_program).results()
raw_counts = (site_res.final_bitstrings if hasattr(site_res, "final_bitstrings")
              else site_res[0].final_bitstrings)

site_counts = Counter(raw_counts)
n_shots_s = sum(site_counts.values())

site_df = pd.DataFrame(
    [
        {
            "bitstring": bits,
            "frequency": count / n_shots_s,
            "nodes": [i for i, b in enumerate(bits) if b == "1"],
        }
        for bits, count in site_counts.most_common()
    ]
)
site_df["independent"] = [is_independent(G, nodes) for nodes in site_df["nodes"]]
site_df["revenue_k"] = [set_weight(G, nodes) for nodes in site_df["nodes"]]
print(site_df.head(10).to_string(index=False))
print(f"\nShare of shots that are feasible plans: {site_df.loc[site_df.independent, 'frequency'].sum():.1%}")

#### Question?

What happens if you put the same amplitude value as in Section 2?

## 3.4 Discussion

* **Why weights matter.** The unweighted MIS maximises the *number* of sites; MWIS maximises *value*.
* **Native geometry.** A conflict graph defined by a distance threshold is a unit-disk graph, which is the structure
  neutral-atom hardware natively represents. General graphs require embedding, which can distort the problem.
* **Size.** Ten sites are still an easy. Quantum approaches are interesting for larger instances; the number of atoms
  and the achievable layouts are limited by the device. For big graphs, try decomposing the problem first.
* **Hybrid pattern.** Classical preprocessing (kernelization), quantum sampling, classical validation and repair: exactly the
  structure used here.
* **Backends.** Moving from the local emulator to a remote emulator or a QPU changes only the Backend
  (and requires a Pasqal Cloud account).

---
**Where to go next**

* QoolQit: [documentation](https://docs.pasqal.com/qoolqit/), [repository](https://github.com/pasqal-io/qoolqit)
* MIS solver: [documentation](https://docs.pasqal.com/applicationsolvingtools/mis/), [repository](https://github.com/pasqal-io/maximum-independent-set)
  (GreedyMIS, advanced quantum programming, MIS via QUBO, graph coloring and 3SAT tutorials)
* Pasqal Community Portal: <https://community.pasqal.com/>

**Challenges**

1. Change `RADIUS_KM` to 0.9 and 1.5. How do the conflict graph, the optimum and the greedy gap change?
2. Add a *must-build* site (set its weight very high) and a *forbidden pair* that is far apart but contractually exclusive.
   The second is no longer a unit-disk graph: what does that change for embedding?